<a href="https://colab.research.google.com/github/trefftzc/voronoiWithJulia/blob/main/Exploring_JACC_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring JACC using a GPU
ACC is a library that makes it possible to write parallel code that will run on either GPUs or microprocessors with several cores, withouth having to change the code.

One selects a backend to decide where the code will be executed, either on a CPU with several cores or on a GPU.

In this notebook, the code runs on a GPU.

In [3]:
using Pkg

In [4]:
Pkg.add("JACC")

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
Precompiling packages...
   9615.4 ms  ✓ JACC → CUDAExt
  1 dependency successfully precompiled in 11 seconds. 533 already precompiled.


In [5]:
import JACC

In [6]:
JACC.set_backend("CUDA")

In [7]:
JACC.@init_backend

[ Info: CUDA backend loaded


In [8]:
using JACC
import JACC

# JACC.@init_backend

# Kernel to find the closest seed

function distance(seedX::Int64,seedY::Int64,x::Int64,y::Int64)
    diff_x = seedX - x
    diff_y = seedY - y
    sum_squares = diff_x * diff_x + diff_y * diff_y
    result = sqrt(sum_squares)
    res::Int64 = round(Int64, result)
    return res
end

function closestSeed(x,y,nSeeds,seedsX::Vector{Int64},seedsY::Vector{Int64})
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    return closest_seed
end

function closestSeedDevice(x,y,canvasDevice,nSeeds,seedsX,seedsY)
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    canvasDevice[x,y] = closest_seed
end
JACC.@init_backend
nSeeds = 4
sizeCanvas = 64
canvas = zeros(Core.Int64,(sizeCanvas,sizeCanvas))
seedsX = zeros(Core.Int64,nSeeds)
seedsY = zeros(Core.Int64,nSeeds)
seedsX[1] = 1
seedsX[2] = 64
seedsX[3] = 1
seedsX[4] = 64
seedsY[1] = 1
seedsY[2] = 1
seedsY[3] = 64
seedsY[4] = 64
canvas_dev = JACC.to_device(canvas)
seedsX_dev = JACC.to_device(seedsX)
seedsY_dev = JACC.to_device(seedsY)

# Sequential version
println("Output of the sequential version:\n")
@inbounds begin
  for y = 1:sizeCanvas
    for x = 1:sizeCanvas
        canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
    end
  end
end


# Print the results

for x = 1:sizeCanvas
    for y = 1:sizeCanvas
        print(canvas[x,y] )
    end
    println()
end
# Now the parallel version

JACC.@parallel_for range=(sizeCanvas,sizeCanvas) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments

canvas2 = JACC.to_host(canvas_dev)
println("Output of the parallel version:\n")
for x = 1:sizeCanvas
    for y = 1:sizeCanvas
        print(canvas2[x,y] )
    end
    println()
end
println("Done")

Output of the sequential version:



[ Info: CUDA backend loaded


1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111

Now with a larger canvas.

In [9]:
# voronoi_with_jacc_size_4096

using JACC
import JACC

const SIZE_AREA_P = 4096
# JACC.@init_backend

# Kernel to find the closest seed

function distance(seedX::Int64,seedY::Int64,x::Int64,y::Int64)
    diff_x = seedX - x
    diff_y = seedY - y
    sum_squares = diff_x * diff_x + diff_y * diff_y
    result = sqrt(sum_squares)
    res::Int64 = round(Int64, result)
    return res
end

function closestSeed(x,y,nSeeds,seedsX::Vector{Int64},seedsY::Vector{Int64})
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    return closest_seed
end

function closestSeedDevice(x,y,canvasDevice,nSeeds,seedsX,seedsY)
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    canvasDevice[x,y] = closest_seed
end

JACC.@init_backend
nSeeds = 4
sizeCanvas = SIZE_AREA_P
canvas = zeros(Core.Int64,(sizeCanvas,sizeCanvas))
seedsX = zeros(Core.Int64,nSeeds)
seedsY = zeros(Core.Int64,nSeeds)
seedsX[1] = 1
seedsX[2] = SIZE_AREA_P
seedsX[3] = 1
seedsX[4] = SIZE_AREA_P
seedsY[1] = 1
seedsY[2] = 1
seedsY[3] = SIZE_AREA_P
seedsY[4] = SIZE_AREA_P
canvas_dev = JACC.to_device(canvas)
seedsX_dev = JACC.to_device(seedsX)
seedsY_dev = JACC.to_device(seedsY)
# Sequential version
@time begin
  @inbounds begin
    for y = 1:sizeCanvas
        for x = 1:sizeCanvas
            canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
        end
    end
  end
end

@time begin
  @inbounds begin
    for y = 1:sizeCanvas
        for x = 1:sizeCanvas
            canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
        end
    end
  end
end

# Now the parallel version
@time begin
    JACC.@parallel_for range=(sizeCanvas,sizeCanvas) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments
end
canvas2 = JACC.to_host(canvas_dev)


@time begin
    JACC.@parallel_for range=(sizeCanvas,sizeCanvas) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments
end
canvas2 = JACC.to_host(canvas_dev)

# Compare the results
outputs_match = true
for y = 1:sizeCanvas
    for x = 1:sizeCanvas
        if canvas[x,y] != canvas2[x,y]
            println("Mismatch at (y): canvas = (canvas2[x,y])")
            global
            outputs_match = false
        end
    end
end

if outputs_match
    println("Outputs match!")
else
    println("Outputs do not match.")
end

[ Info: CUDA backend loaded


  4.432543 seconds (46.18 M allocations: 961.436 MiB, 16.78% gc time, 1.04% compilation time)
  2.766503 seconds (46.16 M allocations: 960.484 MiB, 4.68% gc time)
  0.288364 seconds (143.03 k allocations: 9.350 MiB, 15.56% compilation time: <1% of which was recompilation)
  0.026077 seconds (440 allocations: 15.609 KiB)
Outputs match!
